Some hyper parameters have been reduced due to computer limitiations, as I do  not have access to strong gpus only a good cpu.

In [ ]:
import os, random, time, itertools, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms, models
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.neighbors import NearestNeighbors

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
 
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = "./typiclust_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
CFG = {
    # SimCLR self-supervised pre-training
    "simclr_epochs"     : 100,   # paper: 500  [↓5×]  done this to reduce time because of comp constraint
    "simclr_lr"         : 0.4,
    "simclr_batch"      : 512,
    "simclr_temperature": 0.5,
    "simclr_proj_dim"   : 128,
    "simclr_momentum"   : 0.9,
    "simclr_wd"         : 1e-4,
 
    # Framework A – fully supervised classifier
    "sup_epochs"        : 50,    # paper: 200  [↓4×]
    "sup_lr"            : 0.025,
    "sup_batch"         : 128,
    "sup_momentum"      : 0.9,
 
    # Framework B – linear probe on SimCLR embeddings
    "lp_epochs"         : 30,    # paper: 100  [↓3×]
    "lp_lr"             : 2.5,
    "lp_batch"          : 256,
 
    # Framework C – semi-supervised (FixMatch-lite)
    "ssl_iters"         : 20_000,  # paper: 400 000  [↓20×]
    "ssl_lr"            : 0.03,
    "ssl_batch_l"       : 64,
    "ssl_batch_u"       : 64,
    "ssl_threshold"     : 0.95,
    "ssl_wd"            : 5e-4,
 
    # Active-Learning settings
    "al_budget"         : 10,    # initial labeled pool = 10 (1 per class on avg)
    "al_iterations"     : 3,     # grow to 10 / 20 / 30 labels
    "typi_k"            : 20,    # k-NN neighbours for typicality
    "max_clusters"      : 500,   # cap on K-Means clusters
}

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
 
tf_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_weak = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_strong = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
tf_simclr = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
 
DATA_ROOT = "./data"
os.makedirs(DATA_ROOT, exist_ok=True)
 
print(f"Device : {DEVICE}")
print(f"Saves  : {SAVE_DIR}")
print("Cell 0 ready.")

In [ ]:
raw_train = datasets.CIFAR10(DATA_ROOT, train=True,  download=True)
raw_test  = datasets.CIFAR10(DATA_ROOT, train=False, download=True)
 
N_TRAIN      = len(raw_train)  
N_TEST       = len(raw_test) 
N_CLS        = 10
ALL_TRAIN_IDX = list(range(N_TRAIN))
 
class PlainDataset(Dataset):
    def __init__(self, raw, transform):
        self.raw = raw
        self.tf  = transform
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        img, label = self.raw[i]
        return self.tf(img), label
 
test_loader = DataLoader(
    PlainDataset(raw_test, tf_plain),
    batch_size=512, shuffle=False, num_workers=2, pin_memory=True,
)
print(f"Cell 1 done  train={N_TRAIN}  test={N_TEST}")


In [ ]:
SIMCLR_CKPT = os.path.join(SAVE_DIR, "simclr_model.pt")
EMBED_PATH   = os.path.join(SAVE_DIR, "embeddings.npy")
 
class SimCLRModel(nn.Module):
    """ResNet-18 backbone + MLP projector, adapted for 32×32 CIFAR."""
    def __init__(self, proj_dim: int = 128):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.backbone.maxpool = nn.Identity()
        feat_dim = self.backbone.fc.in_features  # 512
        self.backbone.fc = nn.Identity()
        self.projector = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feat_dim, proj_dim),
        )
 
    def forward(self, x, return_feat: bool = False):
        feat = self.backbone(x)
        if return_feat:
            return F.normalize(feat, dim=1)
        return F.normalize(self.projector(feat), dim=1)
 
 
class NTXentLoss(nn.Module):
    """Normalised temperature-scaled cross-entropy (NT-Xent) loss."""
    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.T = temperature
 
    def forward(self, z1, z2):
        N = z1.size(0)
        z = torch.cat([z1, z2], dim=0)           # (2N, D)
        sim = torch.mm(z, z.t()) / self.T         # (2N, 2N)
        # mask out self-similarity
        sim.masked_fill_(
            torch.eye(2 * N, dtype=torch.bool, device=z.device),
            float("-inf"),
        )
        pos = torch.cat([torch.arange(N, 2 * N), torch.arange(N)]).to(z.device)
        return F.cross_entropy(sim, pos)
 
 
class SimCLRDataset(Dataset):
    def __init__(self, raw):
        self.raw = raw
    def __len__(self):
        return len(self.raw)
    def __getitem__(self, i):
        img, _ = self.raw[i]
        return tf_simclr(img), tf_simclr(img)
 
 
def train_simclr():
    if os.path.exists(SIMCLR_CKPT) and os.path.exists(EMBED_PATH):
        print("SimCLR checkpoint found – skipping training. Running Cell 3.")
        return
 
    epochs = CFG["simclr_epochs"]
    print(f"Training SimCLR for {epochs} epochs ...")
    t0 = time.time()
 
    loader = DataLoader(
        SimCLRDataset(raw_train),
        batch_size=CFG["simclr_batch"], shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True,
    )
    model     = SimCLRModel(CFG["simclr_proj_dim"]).to(DEVICE)
    criterion = NTXentLoss(CFG["simclr_temperature"])
    optimizer = optim.SGD(
        model.parameters(),
        lr=CFG["simclr_lr"],
        momentum=CFG["simclr_momentum"],
        weight_decay=CFG["simclr_wd"],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
 
    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE), v2.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(v1), model(v2))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if ep % 10 == 0 or ep == 1:
            avg_loss = total_loss / len(loader)
            lr_now   = scheduler.get_last_lr()[0]
            elapsed  = time.time() - t0
            print(f"  Epoch {ep:3d}/{epochs}  loss={avg_loss:.4f}  lr={lr_now:.4f}  elapsed={elapsed:.0f}s")
 
    torch.save(model.state_dict(), SIMCLR_CKPT)
    print(f"Model saved -> {SIMCLR_CKPT}")
 
    # Extracting the embeddibgs here and then i will save 
    print("Extracting embeddings ...")
    model.eval()
    embed_loader = DataLoader(
        PlainDataset(raw_train, tf_plain),
        batch_size=512, shuffle=False, num_workers=2, pin_memory=True,
    )
    feats = []
    with torch.no_grad():
        for x, _ in embed_loader:
            feats.append(model(x.to(DEVICE), return_feat=True).cpu().numpy())
    embs = np.concatenate(feats, axis=0)
    np.save(EMBED_PATH, embs)
    print(f"Embeddings saved -> {EMBED_PATH}  shape={embs.shape}  total={time.time()-t0:.0f}s")
 
 
train_simclr()

In [ ]:
def load_simclr():
    assert os.path.exists(SIMCLR_CKPT), f"Missing checkpoint: {SIMCLR_CKPT} – run Cell 2 first."
    assert os.path.exists(EMBED_PATH),  f"Missing embeddings: {EMBED_PATH}  – run Cell 2 first."
    model = SimCLRModel(CFG["simclr_proj_dim"])
    model.load_state_dict(torch.load(SIMCLR_CKPT, map_location=DEVICE))
    model = model.to(DEVICE).eval()
    embs  = np.load(EMBED_PATH)
    print(f"SimCLR loaded  embedding shape={embs.shape}  device={DEVICE}")
    return model, embs
 
 
simclr_model, embeddings = load_simclr()
 

In [ ]:
TYPI_PATH = os.path.join(SAVE_DIR, "typi_labeled.json")
RAND_PATH = os.path.join(SAVE_DIR, "rand_labeled.json")
 
 
def compute_typicality(embs_subset: np.ndarray, k: int = 20) -> np.ndarray:
    """
    Typicality = inverse of mean k-NN distance in embedding space.
    Eq. (4) in Hacohen et al. 2022.
    """
    k_eff = min(k + 1, len(embs_subset))   # +1 because point is its own neighbour
    nn = NearestNeighbors(n_neighbors=k_eff, metric="euclidean", algorithm="auto")
    nn.fit(embs_subset)
    dists, _ = nn.kneighbors(embs_subset)
    mean_dists = dists[:, 1:].mean(axis=1) 
    mean_dists = np.clip(mean_dists, 1e-10, None)
    return 1.0 / mean_dists
 
 
def typiclust_query(embeddings_all: np.ndarray,
                    existing_labeled: list,
                    budget: int,
                    max_clusters: int = 500,
                    k: int = 20) -> list:

    n_total  = len(embeddings_all)
    K        = min(len(existing_labeled) + budget, max_clusters)
    K        = max(K, budget) 
 
    # 1: cluster
    if K <= 50:
        km = KMeans(n_clusters=K, random_state=SEED, n_init=10)
    else:
        km = MiniBatchKMeans(n_clusters=K, random_state=SEED, batch_size=1024, n_init=5)
    cluster_ids = km.fit_predict(embeddings_all)   # shape (N,)
 
    labeled_set = set(existing_labeled)
 
    # Step 2: I am now going to find uncovered clusters
    covered_clusters = set(cluster_ids[i] for i in existing_labeled)
    uncovered        = [c for c in range(K) if c not in covered_clusters]
 
    # Step 3: ranking by cluster size, pick best 'budget'
    cluster_sizes  = {c: (cluster_ids == c).sum() for c in uncovered}
    sorted_clusters = sorted(uncovered, key=lambda c: cluster_sizes[c], reverse=True)
    top_clusters    = sorted_clusters[:budget]
 
    queries = []
    for c in top_clusters:
        member_idx = np.where(cluster_ids == c)[0]
        member_idx = [i for i in member_idx if i not in labeled_set]
        if len(member_idx) == 0:
            continue
        member_embs = embeddings_all[member_idx]
        typ_scores  = compute_typicality(member_embs, k=k)
        best_local  = int(np.argmax(typ_scores))
        queries.append(int(member_idx[best_local]))
        labeled_set.add(int(member_idx[best_local]))
 
    return queries
 
 
def build_typiclust_queries():
    if os.path.exists(TYPI_PATH) and os.path.exists(RAND_PATH):
        typi = json.load(open(TYPI_PATH))
        rand = json.load(open(RAND_PATH))
        print(f"Query lists already exist  typi={len(typi)}  rand={len(rand)}")
        return typi, rand
 
    total_labels = CFG["al_budget"] * CFG["al_iterations"]   # 30 total
    typi_labeled: list = []
    rand_labeled: list = []
 
    print(f"Building TPC_RP query list  (budget={CFG['al_budget']}  iters={CFG['al_iterations']})")
    for it in range(1, CFG["al_iterations"] + 1):
        t0 = time.time()
        new_typi = typiclust_query(
            embeddings,
            existing_labeled=typi_labeled,
            budget=CFG["al_budget"],
            max_clusters=CFG["max_clusters"],
            k=CFG["typi_k"],
        )
        typi_labeled.extend(new_typi)
 
        # Random baseline here 
        remaining = list(set(ALL_TRAIN_IDX) - set(rand_labeled))
        new_rand  = random.sample(remaining, CFG["al_budget"])
        rand_labeled.extend(new_rand)
 
        print(f"  Iter {it}: +{len(new_typi)} typi  +{len(new_rand)} rand  "
              f"total_typi={len(typi_labeled)}  elapsed={time.time()-t0:.1f}s")
 
    with open(TYPI_PATH, "w") as f: json.dump(typi_labeled, f)
    with open(RAND_PATH,  "w") as f: json.dump(rand_labeled,  f)
    print(f"Saved -> {TYPI_PATH}  {RAND_PATH}")
    return typi_labeled, rand_labeled
 
 
typi_all, rand_all = build_typiclust_queries()
print(f"TypiClust indices (first 10): {typi_all[:10]}")
print(f"Random    indices (first 10): {rand_all[:10]}")

In [ ]:
RESULTS_A = os.path.join(SAVE_DIR, "results_A.json")
 
 
def build_resnet18() -> nn.Module:
    net = models.resnet18(weights=None)
    net.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    net.maxpool = nn.Identity()
    net.fc      = nn.Linear(net.fc.in_features, N_CLS)
    return net
 
 
class LabeledDataset(Dataset):
    def __init__(self, indices, transform):
        self.data = [(raw_train[i][0], raw_train[i][1]) for i in indices]
        self.tf   = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        img, label = self.data[i]
        return self.tf(img), label
 
 
@torch.no_grad()
def evaluate(net: nn.Module) -> float:
    net.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += (net(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return round(100.0 * correct / total, 2)
 
 
def train_supervised(labeled_indices: list, tag: str = "") -> float:
    print(f"  [Supervised {tag}]  n_labeled={len(labeled_indices)}")
    ds     = LabeledDataset(labeled_indices, tf_train)
    loader = DataLoader(ds, batch_size=CFG["sup_batch"], shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=False)
    net    = build_resnet18().to(DEVICE)
    opt    = optim.SGD(net.parameters(), lr=CFG["sup_lr"],
                       momentum=CFG["sup_momentum"], nesterov=True, weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["sup_epochs"])
 
    for ep in range(CFG["sup_epochs"]):
        net.train()
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            F.cross_entropy(net(x), y).backward()
            opt.step()
        sched.step()
 
    acc = evaluate(net)
    print(f"  [{tag}] acc={acc:.2f}%")
    del net; torch.cuda.empty_cache()
    return acc
 
 
def run_framework_A():
    results = json.load(open(RESULTS_A)) if os.path.exists(RESULTS_A) else {"typiclust": [], "random": []}
    done    = len(results["typiclust"])
    t_all   = json.load(open(TYPI_PATH))
    r_all   = json.load(open(RAND_PATH))
 
    for it in range(1, CFG["al_iterations"] + 1):
        if it <= done:
            print(f"[A] Iteration {it} already saved – skipping.")
            continue
        n = it * CFG["al_budget"]
        print(f"\n[Framework A] Iteration {it}  ({n} labels)")
        acc_t = train_supervised(t_all[:n], tag="TypiClust")
        acc_r = train_supervised(r_all[:n], tag="Random")
        results["typiclust"].append(acc_t)
        results["random"].append(acc_r)
        with open(RESULTS_A, "w") as f: json.dump(results, f, indent=2)
        print(f"[A] TypiClust={acc_t:.2f}%  Random={acc_r:.2f}%  Gain={acc_t - acc_r:+.2f}%")
 
    print(f"[Framework A] Done -> {RESULTS_A}")
    return results
 
 
results_A = run_framework_A()

In [ ]:
RESULTS_B = os.path.join(SAVE_DIR, "results_B.json")
 
 
def get_test_embeddings() -> torch.Tensor:
    simclr_model.eval()
    feats = []
    with torch.no_grad():
        for x, _ in test_loader:
            feats.append(simclr_model(x.to(DEVICE), return_feat=True).cpu())
    return torch.cat(feats, dim=0)
 
 
def train_linear_probe(labeled_indices: list, tag: str = "") -> float:
    print(f"  [Linear Probe {tag}]  n_labeled={len(labeled_indices)}")
    X_tr = torch.tensor(embeddings[labeled_indices], dtype=torch.float32)
    y_tr = torch.tensor([raw_train[i][1] for i in labeled_indices], dtype=torch.long)
    X_te = get_test_embeddings()
    y_te = torch.tensor([raw_test[i][1] for i in range(N_TEST)], dtype=torch.long)
 
    feat_dim = embeddings.shape[1]
    lp  = nn.Linear(feat_dim, N_CLS).to(DEVICE)
    opt = optim.SGD(lp.parameters(), lr=CFG["lp_lr"], momentum=0.9, weight_decay=0.0)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["lp_epochs"])
 
    for _ in range(CFG["lp_epochs"]):
        lp.train()
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), CFG["lp_batch"]):
            xb = X_tr[perm[i : i + CFG["lp_batch"]]].to(DEVICE)
            yb = y_tr[perm[i : i + CFG["lp_batch"]]].to(DEVICE)
            opt.zero_grad()
            F.cross_entropy(lp(xb), yb).backward()
            opt.step()
        sch.step()
 
    lp.eval()
    correct = total = 0
    with torch.no_grad():
        for i in range(0, len(X_te), 512):
            xb = X_te[i : i + 512].to(DEVICE)
            yb = y_te[i : i + 512].to(DEVICE)
            correct += (lp(xb).argmax(1) == yb).sum().item()
            total   += yb.size(0)
    acc = round(100.0 * correct / total, 2)
    print(f"  [{tag}] acc={acc:.2f}%")
    del lp; torch.cuda.empty_cache()
    return acc
 
 
def run_framework_B():
    results = json.load(open(RESULTS_B)) if os.path.exists(RESULTS_B) else {"typiclust": [], "random": []}
    done    = len(results["typiclust"])
    t_all   = json.load(open(TYPI_PATH))
    r_all   = json.load(open(RAND_PATH))
 
    for it in range(1, CFG["al_iterations"] + 1):
        if it <= done:
            print(f"[B] Iteration {it} already saved – skipping.")
            continue
        n = it * CFG["al_budget"]
        print(f"\n[Framework B] Iteration {it}  ({n} labels)")
        acc_t = train_linear_probe(t_all[:n], tag="TypiClust")
        acc_r = train_linear_probe(r_all[:n], tag="Random")
        results["typiclust"].append(acc_t)
        results["random"].append(acc_r)
        with open(RESULTS_B, "w") as f: json.dump(results, f, indent=2)
        print(f"[B] TypiClust={acc_t:.2f}%  Random={acc_r:.2f}%  Gain={acc_t - acc_r:+.2f}%")
 
    print(f"[Framework B] Done -> {RESULTS_B}")
    return results
 
 
results_B = run_framework_B()

In [ ]:
RESULTS_C = os.path.join(SAVE_DIR, "results_C.json")
 
 
class UnlabeledDataset(Dataset):
    def __init__(self, indices):
        self.images = [raw_train[i][0] for i in indices]
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return tf_weak(self.images[i]), tf_strong(self.images[i])
 
 
def train_semi_supervised(labeled_indices: list, tag: str = "") -> float:
    print(f"  [Semi-Supervised {tag}]  n_labeled={len(labeled_indices)}")
    unlabeled_idx = [i for i in ALL_TRAIN_IDX if i not in set(labeled_indices)]
 
    lab_ds  = LabeledDataset(labeled_indices, tf_weak)
    unl_ds  = UnlabeledDataset(unlabeled_idx)
    lab_loader = DataLoader(lab_ds, batch_size=min(CFG["ssl_batch_l"], len(lab_ds)),
                            shuffle=True, num_workers=2, drop_last=False)
    unl_loader = DataLoader(unl_ds, batch_size=CFG["ssl_batch_u"],
                            shuffle=True, num_workers=2, drop_last=True)
 
    net = build_resnet18().to(DEVICE)
    opt = optim.SGD(net.parameters(), lr=CFG["ssl_lr"],
                    momentum=0.9, weight_decay=CFG["ssl_wd"], nesterov=True)
 
    lab_iter = itertools.cycle(lab_loader)
    unl_iter = itertools.cycle(unl_loader)
    net.train()
 
    total_iters = CFG["ssl_iters"]
    for step in range(1, total_iters + 1):
        x_l, y_l     = next(lab_iter)
        x_uw, x_us   = next(unl_iter)
        x_l,  y_l    = x_l.to(DEVICE),  y_l.to(DEVICE)
        x_uw, x_us   = x_uw.to(DEVICE), x_us.to(DEVICE)
 
        loss_sup = F.cross_entropy(net(x_l), y_l)
 
        with torch.no_grad():
            probs    = F.softmax(net(x_uw), dim=1)
            max_p, pseudo_y = probs.max(dim=1)
            mask     = max_p.ge(CFG["ssl_threshold"]).float()
 
        loss_unl = (F.cross_entropy(net(x_us), pseudo_y, reduction="none") * mask).mean()
        opt.zero_grad()
        (loss_sup + loss_unl).backward()
        opt.step()
 
        if step % 5000 == 0:
            acc = evaluate(net); net.train()
            print(f"    step {step}/{total_iters}  acc={acc:.1f}%")
 
    acc = evaluate(net)
    print(f"  [{tag}] final acc={acc:.2f}%")
    del net; torch.cuda.empty_cache()
    return acc
 
 
def run_framework_C():
    results = json.load(open(RESULTS_C)) if os.path.exists(RESULTS_C) else {"typiclust": [], "random": []}
    done    = len(results["typiclust"])
    t_all   = json.load(open(TYPI_PATH))
    r_all   = json.load(open(RAND_PATH))
 
    for it in range(1, CFG["al_iterations"] + 1):
        if it <= done:
            print(f"[C] Iteration {it} already saved – skipping.")
            continue
        n = it * CFG["al_budget"]
        print(f"\n[Framework C] Iteration {it}  ({n} labels)")
        acc_t = train_semi_supervised(t_all[:n], tag="TypiClust")
        acc_r = train_semi_supervised(r_all[:n], tag="Random")
        results["typiclust"].append(acc_t)
        results["random"].append(acc_r)
        with open(RESULTS_C, "w") as f: json.dump(results, f, indent=2)
        print(f"[C] TypiClust={acc_t:.2f}%  Random={acc_r:.2f}%  Gain={acc_t - acc_r:+.2f}%")
 
    print(f"[Framework C] Done -> {RESULTS_C}")
    return results
 
 
results_C = run_framework_C()
 

In [ ]:
def print_summary():
    B = CFG["al_budget"]
    fw_info = [
        ("A – Fully Supervised",               RESULTS_A),
        ("B – Self-Supervised Embedding",       RESULTS_B),
        ("C – Semi-Supervised (FixMatch-lite)", RESULTS_C),
    ]
    print("\n" + "=" * 65)
    print("  TYPICLUST TPC_RP vs RANDOM  –  CIFAR-10 RESULTS")
    print("=" * 65)
    for fw_name, path in fw_info:
        if not os.path.exists(path):
            print(f"\n[{fw_name}]  not yet run")
            continue
        res = json.load(open(path))
        print(f"\n[{fw_name}]")
        print(f"  {'Labels':>8}  {'TypiClust':>10}  {'Random':>8}  {'Gain':>8}")
        print(f"  {'-' * 44}")
        for i, (t, r) in enumerate(zip(res["typiclust"], res["random"])):
            n = (i + 1) * B
            print(f"  {n:>8}  {t:>9.2f}%  {r:>7.2f}%  {t - r:>+7.2f}%")
 
    print("\n" + "=" * 65)
    print("  Checkpoint files in:", SAVE_DIR)
    for fname in sorted(os.listdir(SAVE_DIR)):
        size = os.path.getsize(os.path.join(SAVE_DIR, fname)) / 1e6
        print(f"    {fname:<40} {size:.1f} MB")
    print("=" * 65)
 
 
print_summary()